# T1.6 — Statistical Robustness Audit

**Purpose:** analysis-only robustness audit on the locked 30-model primary experiment.

This notebook does **not** replace the locked primary Holm-corrected H1/H2 analysis. It provides three secondary checks:

1. Manual balanced-design variance decomposition: between-ρ signal versus within-ρ replicate/seed variability.
2. Mixed-effects model as a **cross-check only**; the manual decomposition remains the authoritative calculation for this balanced one-way design.
3. Heteroscedasticity across ρ and deterministic, ρ-stratified cluster bootstrap CIs for the linear/quadratic shape parameters used in T1.2.

The source table is the existing `model_endpoints_final_checkpoint.csv` generated by the frozen primary analysis pipeline.

In [1]:
# ================================================================
# CELL 1 — IMPORTS AND LOCKED ANALYSIS CONSTANTS
# ================================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

SEED = 20260924
N_BOOTSTRAP = 20_000
RHO_LEVELS = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
                
ENDPOINTS = [
    'delta_z_hair',
    'D',
    'D_AUROC',
    'xai_enrichment',
]

REQUIRED_META = ['model', 'rho', 'replicate', 'checkpoint']

print('T1.6 — STATISTICAL ROBUSTNESS AUDIT')
print(f'Seed: {SEED}')
print(f'Bootstrap draws: {N_BOOTSTRAP:,}')

T1.6 — STATISTICAL ROBUSTNESS AUDIT
Seed: 20260924
Bootstrap draws: 20,000


In [2]:
# ================================================================
# CELL 2 — LOAD THE FROZEN 30-MODEL PRIMARY ENDPOINT TABLE
# ================================================================
# Do NOT rerun model training or Stage 2/3 extraction here.
# The primary analysis notebook already created this table.

CANDIDATES = [
    Path('/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/results/model_endpoints_final_checkpoint.csv'),
    Path('/kaggle/working/results/model_endpoints_final_checkpoint.csv'),
    Path('/kaggle/working/model_endpoints_final_checkpoint.csv'),
    Path('results/model_endpoints_final_checkpoint.csv'),
    Path('model_endpoints_final_checkpoint.csv'),
]

CSV_PATH = next((p for p in CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        'model_endpoints_final_checkpoint.csv was not found. '
        'Place the frozen Stage-4 CSV in the working/input path.'
    )

df = pd.read_csv(CSV_PATH)

assert len(df) == 30, f'Expected 30 primary models, found {len(df)}'
assert df['model'].nunique() == 30, 'Model identifiers are not unique.'
assert sorted(df['rho'].unique().tolist()) == RHO_LEVELS.tolist()
assert all(df.groupby('rho').size().to_numpy() == 5)
assert set(df['checkpoint'].astype(str)) == {'final'} or len(df['checkpoint'].unique()) == 1

missing = [c for c in REQUIRED_META + ENDPOINTS if c not in df.columns]
assert not missing, f'Missing required columns: {missing}'

for c in ENDPOINTS:
    assert np.isfinite(df[c].to_numpy(dtype=float)).all(), f'Non-finite values in {c}'

print(f'Loaded: {CSV_PATH}')
print(f'Rows: {len(df)}')
print('✓ 6 rho levels × 5 replicates = 30 models')
print('✓ Required T1.6 endpoints present and finite')

display(df[REQUIRED_META + ENDPOINTS].sort_values(['rho', 'replicate']).reset_index(drop=True))

Loaded: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/results/model_endpoints_final_checkpoint.csv
Rows: 30
✓ 6 rho levels × 5 replicates = 30 models
✓ Required T1.6 endpoints present and finite


,model,rho,replicate,checkpoint,delta_z_hair,D,D_AUROC,xai_enrichment
0,rho_0.0_rep_1,0.0,1,final,1.732511,0.021146,0.032008,7.265268
1,rho_0.0_rep_2,0.0,2,final,1.793227,0.020447,0.034044,7.038029
2,rho_0.0_rep_3,0.0,3,final,1.904324,0.025341,0.035462,7.523387
3,rho_0.0_rep_4,0.0,4,final,1.870657,0.020972,0.035110,7.873954
4,rho_0.0_rep_5,0.0,5,final,1.563364,0.019748,0.028267,6.742265
5,rho_0.2_rep_1,0.2,1,final,4.803006,0.087732,0.112832,8.855382
6,rho_0.2_rep_2,0.2,2,final,5.166609,0.095945,0.124537,9.887817
7,rho_0.2_rep_3,0.2,3,final,5.292174,0.082139,0.113141,10.181834
8,rho_0.2_rep_4,0.2,4,final,5.245159,0.093149,0.118447,10.214259
9,rho_0.2_rep_5,0.2,5,final,4.844580,0.090353,0.115939,10.342756


## A. Manual balanced-design variance decomposition

For each endpoint, the total sum of squares is decomposed into between-ρ and within-ρ components. The **within-ρ component is the replicate/seed instability component**. Because the design has exactly five replicates at each of six ρ levels, this decomposition is exact for the observed balanced one-way design.

This is the primary T1.6 variance calculation. It is not replaced by `MixedLM`.

In [3]:
# ================================================================
# CELL 3 — MANUAL BETWEEN/WITHIN VARIANCE DECOMPOSITION
# ================================================================

variance_rows = []

for endpoint in ENDPOINTS:
    grand_mean = df[endpoint].mean()
    group_means = df.groupby('rho')[endpoint].mean()
    n_per_group = df.groupby('rho')[endpoint].count()

    ss_between = float(((group_means - grand_mean) ** 2 * n_per_group).sum())
    df_between = len(RHO_LEVELS) - 1
    ms_between = ss_between / df_between

    ss_within = float(sum(
        ((df.loc[df['rho'] == r, endpoint] - group_means[r]) ** 2).sum()
        for r in RHO_LEVELS
    ))
    df_within = len(df) - len(RHO_LEVELS)
    ms_within = ss_within / df_within

    ss_total = ss_between + ss_within
    between_fraction = ss_between / ss_total if ss_total > 0 else np.nan
    within_fraction = ss_within / ss_total if ss_total > 0 else np.nan
    f_ratio = ms_between / ms_within if ms_within > 0 else np.inf

    # One-way ANOVA ICC-style estimator for a balanced design.
    # This is reported separately from the simple variance fractions.
    k = 5
    icc_anova = ((ms_between - ms_within) /
                (ms_between + (k - 1) * ms_within)
                if (ms_between + (k - 1) * ms_within) > 0 else np.nan)

    variance_rows.append({
        'endpoint': endpoint,
        'ss_between_rho': ss_between,
        'ss_within_rho': ss_within,
        'ms_between_rho': ms_between,
        'ms_within_rho': ms_within,
        'between_rho_variance_fraction': between_fraction,
        'within_rho_variance_fraction': within_fraction,
        'F_ratio_between_over_within': f_ratio,
        'icc_anova_style': icc_anova,
    })

variance_df = pd.DataFrame(variance_rows)
display(variance_df.round(6))

print('\nInterpretation rule:')
print('  between-rho fraction = observed variation attributable to separation of rho group means')
print('  within-rho fraction  = observed replicate/seed variability around those group means')
print('  ICC-style value      = ANOVA variance-component cross-reference, not a replacement for the decomposition')

,endpoint,ss_between_rho,ss_within_rho,ms_between_rho,ms_within_rho,between_rho_variance_fraction,within_rho_variance_fraction,F_ratio_between_over_within,icc_anova_style
0,delta_z_hair,345.493175,1.425630,69.098635,0.059401,0.995891,0.004109,1163.252351,0.995716
1,D,0.109219,0.000991,0.021844,0.000041,0.991010,0.008990,529.148116,0.990622
2,D_AUROC,0.456302,0.000991,0.091260,0.000041,0.997832,0.002168,2209.122830,0.997741
3,xai_enrichment,298.701739,7.937101,59.740348,0.330713,0.974116,0.025884,180.641313,0.972920



Interpretation rule:
  between-rho fraction = observed variation attributable to separation of rho group means
  within-rho fraction  = observed replicate/seed variability around those group means
  ICC-style value      = ANOVA variance-component cross-reference, not a replacement for the decomposition


## B. Mixed-effects cross-check

The mixed-effects model is deliberately secondary. It uses `rho` as a fixed effect and replicate index as a random intercept, matching the intended conceptual model. With only five replicate groups, convergence or singular-fit warnings are possible; if that happens, the manual balanced-design decomposition above remains the authoritative calculation.

In [4]:
# ================================================================
# CELL 4 — MIXEDLM CROSS-CHECK ONLY
# ================================================================

mixedlm_rows = []

try:
    import statsmodels.formula.api as smf
    HAVE_STATSMODELS = True
except ImportError:
    HAVE_STATSMODELS = False
    print('statsmodels unavailable; MixedLM cross-check skipped.')

if HAVE_STATSMODELS:
    for endpoint in ENDPOINTS:
        mdf = df[['rho', 'replicate', endpoint]].copy()
        try:
            model = smf.mixedlm(
                f'{endpoint} ~ rho',
                mdf,
                groups=mdf['replicate'],
                re_formula='1'
            )
            fit = model.fit(reml=True, method='lbfgs', maxiter=2000, disp=False)

            random_var = float(fit.cov_re.iloc[0, 0])
            residual_var = float(fit.scale)
            mixedlm_rows.append({
                'endpoint': endpoint,
                'converged': bool(getattr(fit, 'converged', True)),
                'rho_fixed_effect': float(fit.params.get('rho', np.nan)),
                'random_intercept_variance': random_var,
                'residual_variance': residual_var,
                'random_to_residual_ratio': random_var / residual_var if residual_var > 0 else np.inf,
                'warning': ''
            })
        except Exception as exc:
            mixedlm_rows.append({
                'endpoint': endpoint,
                'converged': False,
                'rho_fixed_effect': np.nan,
                'random_intercept_variance': np.nan,
                'residual_variance': np.nan,
                'random_to_residual_ratio': np.nan,
                'warning': repr(exc)
            })

mixedlm_df = pd.DataFrame(mixedlm_rows)
display(mixedlm_df.round(6))

print('\nMixedLM is a cross-check only. The manual decomposition is retained even if MixedLM fails or is singular.')

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/d

,endpoint,converged,rho_fixed_effect,random_intercept_variance,residual_variance,random_to_residual_ratio,warning
0,delta_z_hair,True,9.849509,0.0,0.262627,0.0,
1,D,True,0.174170,0.0,0.000138,0.0,
2,D_AUROC,True,0.357482,0.0,0.000355,0.0,
3,xai_enrichment,True,9.073338,0.0,0.625472,0.0,



MixedLM is a cross-check only. The manual decomposition is retained even if MixedLM fails or is singular.


## C. Heteroscedasticity across ρ

Two complementary diagnostics are reported:

- Levene's test across the six ρ groups.
- Spearman association between ρ and the six within-ρ standard deviations.

These diagnostics describe whether replicate variability changes with training association strength; they do not alter the locked primary test.

In [5]:
# ================================================================
# CELL 5 — HETEROSCEDASTICITY CHECK
# ================================================================

hetero_rows = []

for endpoint in ENDPOINTS:
    groups = [df.loc[df['rho'] == r, endpoint].to_numpy(dtype=float) for r in RHO_LEVELS]
    sds = np.array([g.std(ddof=1) for g in groups])
    lev_stat, lev_p = stats.levene(*groups, center='median')
    rho_sd = stats.spearmanr(RHO_LEVELS, sds)

    row = {
        'endpoint': endpoint,
        'levene_stat': float(lev_stat),
        'levene_p': float(lev_p),
        'spearman_rho_vs_within_sd': float(rho_sd.statistic),
        'spearman_p': float(rho_sd.pvalue),
    }
    for r, sd in zip(RHO_LEVELS, sds):
        row[f'sd_rho_{r:.1f}'] = float(sd)
    hetero_rows.append(row)

hetero_df = pd.DataFrame(hetero_rows)
display(hetero_df.round(6))


,endpoint,levene_stat,levene_p,spearman_rho_vs_within_sd,spearman_p,sd_rho_0.0,sd_rho_0.2,sd_rho_0.4,sd_rho_0.6,sd_rho_0.8,sd_rho_1.0
0,delta_z_hair,1.520595,0.220742,0.200000,0.704000,0.134888,0.229932,0.228879,0.226043,0.082251,0.418447
1,D,0.680692,0.642389,0.714286,0.110787,0.002198,0.005297,0.007713,0.006588,0.007777,0.007171
2,D_AUROC,0.826993,0.542978,0.942857,0.004805,0.002958,0.004803,0.004053,0.005792,0.007217,0.010677
3,xai_enrichment,0.582228,0.713268,-0.028571,0.957155,0.435768,0.605296,0.378911,0.422396,0.233209,1.025485


## D. ρ-stratified cluster bootstrap for T1.2 shape parameters

T1.2 fit linear and quadratic dose-response models. The bootstrap below resamples **replicates within each ρ level**, keeping all six ρ levels represented in every draw. This preserves the experimental structure better than treating all 30 models as exchangeable rows.

For each endpoint, the notebook reports deterministic 95% percentile CIs for:

- linear intercept β₀
- linear slope β₁
- quadratic intercept β₀
- quadratic linear coefficient β₁
- quadratic curvature coefficient β₂

The point estimates use the original 30 observations. These CIs supplement, rather than replace, the T1.2 shape tests.

In [6]:
# ================================================================
# CELL 6 — STRATIFIED CLUSTER BOOTSTRAP OF LINEAR/QUADRATIC SHAPE
# ================================================================

def poly_beta(x, y, degree):
    X = np.column_stack([x ** k for k in range(degree + 1)])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return beta

bootstrap_rows = []
rng = np.random.default_rng(SEED)

for endpoint in ENDPOINTS:
    x = df['rho'].to_numpy(dtype=float)
    y = df[endpoint].to_numpy(dtype=float)

    beta1_point = poly_beta(x, y, 1)
    beta2_point = poly_beta(x, y, 2)

    boot_linear = np.empty((N_BOOTSTRAP, 2), dtype=float)
    boot_quad = np.empty((N_BOOTSTRAP, 3), dtype=float)

    # Resample the five replicate observations within each rho stratum.
    strata = {
        float(r): df.loc[df['rho'] == r, endpoint].to_numpy(dtype=float)
        for r in RHO_LEVELS
    }

    for b in range(N_BOOTSTRAP):
        xb = []
        yb = []
        for r in RHO_LEVELS:
            vals = strata[float(r)]
            sampled = rng.choice(vals, size=len(vals), replace=True)
            xb.extend([r] * len(vals))
            yb.extend(sampled.tolist())
                
        xb = np.asarray(xb, dtype=float)
        yb = np.asarray(yb, dtype=float)
        boot_linear[b] = poly_beta(xb, yb, 1)
        boot_quad[b] = poly_beta(xb, yb, 2)

    def ci(arr, j):
        return np.percentile(arr[:, j], [2.5, 97.5])

    for model_name, point, boot in [
        ('linear', beta1_point, boot_linear),
        ('quadratic', beta2_point, boot_quad),
    ]:
        for j, parameter in enumerate(['beta0', 'beta1'] if model_name == 'linear' else ['beta0', 'beta1', 'beta2']):
            lo, hi = ci(boot, j)
            bootstrap_rows.append({
                'endpoint': endpoint,
                'model': model_name,
                'parameter': parameter,
                'point_estimate': float(point[j]),
                'ci95_low': float(lo),
                'ci95_high': float(hi),
                'n_bootstrap': N_BOOTSTRAP,
                'seed': SEED,
                'resampling_scheme': '5 replicates within each of 6 rho strata, replacement',
            })

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df.round(8))


,endpoint,model,parameter,point_estimate,ci95_low,ci95_high,n_bootstrap,seed,resampling_scheme
0,delta_z_hair,linear,beta0,2.282351,2.162176,2.395604,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
1,delta_z_hair,linear,beta1,9.849509,9.598944,10.119376,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
2,delta_z_hair,quadratic,beta0,2.225891,2.108116,2.334193,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
3,delta_z_hair,quadratic,beta1,10.272961,9.520310,11.026237,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
4,delta_z_hair,quadratic,beta2,-0.423451,-1.300480,0.504663,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
5,D,linear,beta0,0.037038,0.034369,0.039598,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
6,D,linear,beta1,0.174170,0.169112,0.179672,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
7,D,quadratic,beta0,0.029035,0.026824,0.031354,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
8,D,quadratic,beta1,0.234198,0.215643,0.251684,20000,20260924,"5 replicates within each of 6 rho strata, repl..."
9,D,quadratic,beta2,-0.060028,-0.078957,-0.039618,20000,20260924,"5 replicates within each of 6 rho strata, repl..."


In [7]:
# ================================================================
# CELL 7 — SAVE T1.6 RESULTS + REPRODUCIBILITY MANIFEST
# ================================================================

OUT = Path('/kaggle/working/t1_6_results')
OUT.mkdir(parents=True, exist_ok=True)

variance_df.to_csv(OUT / 'T1_6_variance_decomposition.csv', index=False)
hetero_df.to_csv(OUT / 'T1_6_heteroscedasticity.csv', index=False)
bootstrap_df.to_csv(OUT / 'T1_6_shape_parameter_bootstrap_CIs.csv', index=False)
if HAVE_STATSMODELS:
    mixedlm_df.to_csv(OUT / 'T1_6_mixedlm_crosscheck.csv', index=False)

manifest = {
    'experiment': 'T1.6',
    'description': 'Statistical robustness audit on locked 30-model primary endpoints',
    'input_file': str(CSV_PATH),
    'n_models': int(len(df)),
    'rho_levels': RHO_LEVELS.tolist(),
    'replicates_per_rho': 5,
    'endpoints': ENDPOINTS,
    'bootstrap_draws': N_BOOTSTRAP,
    'bootstrap_seed': SEED,
    'bootstrap_scheme': 'resample replicates within each rho level; retain all six rho levels',
    'mixedlm_role': 'cross-check only; manual balanced-design decomposition is primary',
    'primary_h1_h2_unchanged': True,
}

with open(OUT / 'T1_6_AUDIT_MANIFEST.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('T1.6 COMPLETE')
print(f'Results written to: {OUT}')
print('Files:')
for p in sorted(OUT.iterdir()):
    print(' -', p.name)


T1.6 COMPLETE
Results written to: /kaggle/working/t1_6_results
Files:
 - T1_6_AUDIT_MANIFEST.json
 - T1_6_heteroscedasticity.csv
 - T1_6_mixedlm_crosscheck.csv
 - T1_6_shape_parameter_bootstrap_CIs.csv
 - T1_6_variance_decomposition.csv


## T1.6 reporting rule

Do not alter the locked primary H1/H2 conclusions based on this audit. Report T1.6 as a robustness characterization of replicate/seed variability, variance stability across ρ, and uncertainty around the T1.2 shape coefficients. If MixedLM fails to converge or is singular, report that transparently and retain the manual balanced-design decomposition as the primary variance calculation.